# Bab 5 — 5.11 Implementasi Python: Decision Tree & Pruning
Modul lengkap: Praktikum 5.1 – 5.6. Dataset: Breast Cancer Wisconsin (569 pasien, 30 fitur).

## Praktikum 5.1 — Persiapan Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

## Praktikum 5.2 — Import Dataset
Dataset dimuat sebagai DataFrame agar nama fiturnya terbaca. Kelas 0 = *malignant*, 1 = *benign*.

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print("Ukuran data:", X.shape)
print("Distribusi kelas:", np.bincount(y))

## Praktikum 5.3 — Preprocessing (Split Data)
Split 80% latih / 20% uji, `stratify=y` menjaga proporsi kelas, `random_state=42` agar reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
    test_size=0.2, random_state=42, stratify=y)

## Praktikum 5.4 — Training Model (dengan dan tanpa Pruning)
Tiga model: tanpa pruning (tumbuh penuh), pre-pruning (`max_depth=4`, `min_samples_leaf=5`), dan post-pruning (Cost Complexity Pruning, alpha diambil dari tengah pruning path).

In [ ]:
# Model tanpa pruning (tumbuh penuh)
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Model dengan pre-pruning
tree_pruned = DecisionTreeClassifier(max_depth=4,
    min_samples_leaf=5, random_state=42)
tree_pruned.fit(X_train, y_train)

# Model dengan post-pruning (Cost Complexity Pruning)
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alpha_terpilih = path.ccp_alphas[len(path.ccp_alphas)//2]  # contoh pemilihan alpha
print("ccp_alpha terpilih:", ccp_alpha_terpilih)
tree_ccp = DecisionTreeClassifier(ccp_alpha=ccp_alpha_terpilih,
    random_state=42)
tree_ccp.fit(X_train, y_train)

## Praktikum 5.5 — Evaluasi
Gap = selisih akurasi train dan test. Gap besar menandakan *overfitting*.

In [ ]:
for nama, model in [("Tanpa Pruning", tree_full),
    ("Pre-Pruning", tree_pruned), ("Post-Pruning (CCP)", tree_ccp)]:
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test = accuracy_score(y_test, model.predict(X_test))
    print(f"{nama}: Akurasi Train={acc_train:.3f}, Akurasi Test={acc_test:.3f}, Gap={acc_train-acc_test:.3f}")

## Praktikum 5.6 — Visualisasi
Pohon hasil pre-pruning: biru = *benign*, oranye = *malignant*, makin pekat makin murni.

In [ ]:
plt.figure(figsize=(14,8))
plot_tree(tree_pruned, feature_names=data.feature_names,
    class_names=data.target_names, filled=True, rounded=True,
    fontsize=7)
plt.title("Decision Tree (max_depth=4, Pre-Pruning)")
plt.show()

## 5.11.1 — Interpretasi
Model tanpa pruning umumnya menunjukkan gap besar antara akurasi train dan test (indikasi *overfitting*), sedangkan model dengan pre-pruning atau post-pruning menunjukkan gap yang lebih kecil, meskipun akurasi train sedikit menurun. Pemilihan strategi pruning terbaik sebaiknya didasarkan pada *validation curve* atau *cross-validation*, bukan sekadar mencoba satu nilai parameter.